In [21]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D,Dropout,Dense
from tensorflow.keras.models import Sequential
from sklearn.metrics import f1_score,classification_report,confusion_matrix

In [8]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.3,
    rotation_range=20,
    shear_range=0.3,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [19]:
train_generator = train_datagen.flow_from_directory(
    'Garbage_Dataset/Garbage_Dataset/train',
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = test_datagen.flow_from_directory(
    'Garbage_Dataset/Garbage_Dataset/validation',
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    'Garbage_Dataset/Garbage_Dataset/test',
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)
print(train_generator.class_indices)
num_class = train_generator.num_classes

Found 2101 images belonging to 6 classes.
Found 446 images belonging to 6 classes.
Found 453 images belonging to 6 classes.
{'battery': 0, 'biological': 1, 'brown-glass': 2, 'cardboard': 3, 'clothes': 4, 'plastic': 5}


In [15]:
resnet_base = ResNet50(
    include_top=False,
    weights=None,
    input_shape=(224,224,3)
)
resnet_base.trainable=False
resnet_model = Sequential()
resnet_model.add(resnet_base)
resnet_model.add(GlobalAveragePooling2D())
resnet_model.add(Dense(512,activation='relu'))
resnet_model.add(Dropout(0.5))
resnet_model.add(Dense(num_class,activation='softmax'))

resnet_model.compile(loss='categorical_crossentropy',metrics=['accuracy'],optimizer='adam')
resnet_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)                │ (None, 7, 7, 2048)          │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_2           │ (None, 2048)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 512)                 │       1,049,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 6)                   │           3,078 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 24,639,878 (93.99 MB)

 Trainable params: 1,052,166 (4.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [20]:
res_history = resnet_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs = 2
)

Epoch 1/2
66/66 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.1880 - loss: 1.7848 - val_accuracy: 0.1480 - val_loss: 1.7860
Epoch 2/2
66/66 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step - accuracy: 0.1913 - loss: 1.7718 - val_accuracy: 0.1704 - val_loss: 1.7621


In [ ]:
res_pred_prob = resnet_model.predict(test_generator)
res_pred = np.argmax(res_pred_prob,axis=1)

true_class  = test_generator.classes

res_f1 = f1_score(true_class,res_pred,average='weighted')
print(res_f1)

 6/15 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step